[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lursmani/DL-Dub/blob/main/colab_dub.ipynb)

# 🎬 Georgian Cartoon Dubbing — Colab (GPU)

Runs the whole pipeline on a free cloud GPU. **Before you start:**
1. **Runtime → Change runtime type → T4 GPU**, then Save.
2. Add three keys in the **🔑 Secrets** panel (left sidebar), toggling *Notebook access* on for each:
   `ELEVENLABS_API_KEY`, `HF_TOKEN`, and at least one translation key: `ANTHROPIC_API_KEY`, `OPENAI_API_KEY`, or `GOOGLE_API_KEY`.
3. Put your episode in Google Drive, e.g. `MyDrive/dubbing/episode.mp4`.
4. Accept the pyannote licenses once (logged in): [diarization-3.1](https://hf.co/pyannote/speaker-diarization-3.1) and [segmentation-3.0](https://hf.co/pyannote/segmentation-3.0).

Then run the cells in order.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi

## 2. Get the pipeline code
Clones the public repo into `/content/app`. No token needed.

In [ ]:
REPO_URL = 'https://github.com/Lursmani/DL-Dub.git'  # public repo — no token needed
# Clone into a fixed dir name so the repo can be renamed without breaking this cell.
!git clone -q $REPO_URL app
%cd app

# --- Fallback: no GitHub. Zip the repo CONTENTS (files at the zip root) as ---
# --- MyDrive/dubbing/dl-dub.zip, then: ---
# from google.colab import drive; drive.mount('/content/drive')
# !mkdir -p app && unzip -q /content/drive/MyDrive/dubbing/dl-dub.zip -d app
# %cd app

## 3. Install dependencies
Takes a few minutes. If a later cell reports CUDA missing, re-run this one.
**After it finishes, do Runtime → Restart session once** (pip upgrades numpy/torch), then continue from the next cell.

In [ ]:
!apt-get -qq install -y ffmpeg > /dev/null
# Colab runs the ML stages, so install BOTH the core and the ML extras.
!pip -q install -r requirements.txt -r requirements-ml.txt
print('deps installed')

## 4. Load your API keys from Colab Secrets

In [ ]:
from google.colab import userdata
required = ['ELEVENLABS_API_KEY', 'HF_TOKEN']
# translation: add at least ONE of these to Secrets
translation = ['ANTHROPIC_API_KEY', 'OPENAI_API_KEY', 'GOOGLE_API_KEY']
def get(k):
    try: return userdata.get(k)
    except Exception: return None
with open('.env', 'w') as f:
    for k in required:
        v = get(k)
        assert v, f'Missing secret: {k} (add it in the 🔑 panel)'
        f.write(f'{k}={v}\n')
    found = [k for k in translation if get(k)]
    assert found, f'Add at least one translation key to Secrets: {translation}'
    for k in found:
        f.write(f'{k}={get(k)}\n')
print('keys written to .env — translation providers:', found)

## 5. Mount Drive & point at your episode

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
VIDEO = '/content/drive/MyDrive/dubbing/episode.mp4'  # <-- your file
import os; assert os.path.exists(VIDEO), f'Not found: {VIDEO}'
print('video ok:', VIDEO)

## 6. Launch the GUI 🎛️

The whole workflow in your browser: analyze → listen to each speaker → assign voices → review/edit translations → see the cost → dub → download.

- Open the printed **`https://….gradio.live`** link in a **new browser tab** (the inline preview is unreliable for live logs).
- **Keep this cell running** — the GUI dies when the cell or runtime stops. If the link dies mid-run, just re-run this cell and click *Refresh from disk*: all progress is on disk, nothing is lost.


In [ ]:
from gui.app import build_app
build_app().launch(share=True, debug=True)

---
# CLI fallback (optional)
Everything below is the pre-GUI manual workflow — the GUI replaces it. Only useful if you prefer driving the pipeline from cells.

### CLI 1 — Config & character voices
Run this once now (with placeholders). After **Pass 1** reveals the speaker labels, come back, fill in a `voice_id` per character, and re-run this cell.

In [ ]:
import yaml

# After Pass 1, map each detected SPEAKER_xx to an ElevenLabs voice.
VOICES = {
    'SPEAKER_00': {'voice_id': 'REPLACE_ME', 'model_id': 'eleven_multilingual_v2'},
    'SPEAKER_01': {'voice_id': 'REPLACE_ME', 'model_id': 'eleven_flash_v2_5'},
}

config = {
    'source_lang': 'nl', 'target_lang': 'ka',
    'whisper_model': 'large-v3',
    'translate_provider': 'anthropic',  # or 'openai' / 'google'
    'translate_model': '',  # blank = provider default
    'chars_per_second': 15,
    'default_tts_model': 'eleven_flash_v2_5',
    'default_voice_id': 'REPLACE_ME',   # fallback for unmapped speakers
    'output_format': 'mp3_44100_128',
    'voices': VOICES,
    'background_gain_db': -3.0,
    'max_speedup': 1.30, 'min_slowdown': 1.0,  # 1.0 = never slow speech to fill a slot
}
yaml.safe_dump(config, open('config.yaml', 'w'), allow_unicode=True, sort_keys=False)
print(open('config.yaml').read())

### CLI 2 — Pass 1: extract, separate, transcribe & detect speakers
The slow GPU stages. When done, note which `SPEAKER_xx` is which character, then update the voices in cell 6 and re-run cell 6.

In [ ]:
!python dub.py run "$VIDEO" --stop-after transcribe

import glob, json
m = json.load(open(glob.glob('work/*/manifest.json')[0], encoding='utf-8'))
print('\n--- detected lines (first 20) ---')
for s in m['segments'][:20]:
    print(f"[{s['speaker']}] {s['start']:.1f}s: {s['text_src']}")

### CLI 3 — Pass 2: translate, preview cost, dub & splice
Make sure you re-ran cell 6 with real voice IDs first. This translates, prints the spend estimate, then synthesizes and rebuilds the video.

In [ ]:
!python dub.py run "$VIDEO" --start-at translate --stop-after translate
!python dub.py estimate "$VIDEO"

In [ ]:
# Synthesize + assemble + mux. --yes skips the interactive prompt (see estimate above).
!python dub.py run "$VIDEO" --start-at tts --yes

# Save the dubbed video back to Drive.
!cp work/*/*.ka.mp4 /content/drive/MyDrive/dubbing/
print('done — check MyDrive/dubbing/ for the *.ka.mp4')